In [0]:
csv_data = """order_id,customer,product,amount
1001,Ravi,Laptop,55000
1002,Sneha,Mobile,18000
1003,Arjun,Tablet,twenty thousand
1004,Divya,Headphones
1005,Karthik,Camera,32000,Express
1006,Meera,Watch,9500"""

base_path = "/Volumes/main/default/demo"   # change to your volume path
path = f"{base_path}/orders.csv"

dbutils.fs.put(path, csv_data, True)

Wrote 191 bytes.


True

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer", StringType()),
    StructField("product",  StringType()),
    StructField("amount",   DoubleType()),
    StructField("_corrupt_record", StringType())   # extra column to catch bad rows
])

In [0]:
schema_clean== StructType([
    StructField("order_id", IntegerType()),
    StructField("customer", StringType()),
    StructField("product",  StringType()),
    StructField("amount",   DoubleType())
])

True

In [0]:
df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(path)
)

df.show(truncate=False)

+--------+--------+----------+-------+---------------------------------+
|order_id|customer|product   |amount |_corrupt_record                  |
+--------+--------+----------+-------+---------------------------------+
|1001    |Ravi    |Laptop    |55000.0|NULL                             |
|1002    |Sneha   |Mobile    |18000.0|NULL                             |
|1003    |Arjun   |Tablet    |NULL   |1003,Arjun,Tablet,twenty thousand|
|1004    |Divya   |Headphones|NULL   |1004,Divya,Headphones            |
|1005    |Karthik |Camera    |32000.0|1005,Karthik,Camera,32000,Express|
|1006    |Meera   |Watch     |9500.0 |NULL                             |
+--------+--------+----------+-------+---------------------------------+



In [0]:
df_permissive = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(schema)
    .csv(path)
)

df_permissive.show(truncate=False)

+--------+--------+----------+-------+---------------------------------+
|order_id|customer|product   |amount |_corrupt_record                  |
+--------+--------+----------+-------+---------------------------------+
|1001    |Ravi    |Laptop    |55000.0|NULL                             |
|1002    |Sneha   |Mobile    |18000.0|NULL                             |
|1003    |Arjun   |Tablet    |NULL   |1003,Arjun,Tablet,twenty thousand|
|1004    |Divya   |Headphones|NULL   |1004,Divya,Headphones            |
|1005    |Karthik |Camera    |32000.0|1005,Karthik,Camera,32000,Express|
|1006    |Meera   |Watch     |9500.0 |NULL                             |
+--------+--------+----------+-------+---------------------------------+



In [0]:
df_permissive.filter("_corrupt_record IS NOT NULL").show()

+--------+--------+----------+-------+--------------------+
|order_id|customer|   product| amount|     _corrupt_record|
+--------+--------+----------+-------+--------------------+
|    1003|   Arjun|    Tablet|   NULL|1003,Arjun,Tablet...|
|    1004|   Divya|Headphones|   NULL|1004,Divya,Headph...|
|    1005| Karthik|    Camera|32000.0|1005,Karthik,Came...|
+--------+--------+----------+-------+--------------------+



In [0]:
len(df_permissive.filter("_corrupt_record IS NOT NULL").collect())   # 3

3

In [0]:

df_drop = (
    spark.read
    .option("header", True)
    .option("mode", "DROPMALFORMED")
    .schema(schema_clean)
    .csv(path)
)

df_drop.show()

+--------+--------+-------+-------+
|order_id|customer|product| amount|
+--------+--------+-------+-------+
|    1001|    Ravi| Laptop|55000.0|
|    1002|   Sneha| Mobile|18000.0|
|    1006|   Meera|  Watch| 9500.0|
+--------+--------+-------+-------+



In [0]:
df_fail = (
    spark.read
    .option("header", True)
    .option("mode", "FAILFAST")
    .schema(schema_clean)
    .csv(path)
)

df_fail.show()   # error appears here, because Spark is lazy

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-4985718856994674>, line 9
      1 df_fail = (
      2     spark.read
      3     .option("header", True)
   (...)      6     .csv(path)
      7 )
----> 9 df_fail.show()   # error appears here, because Spark is lazy

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1297, in DataFrame.show(self, n, truncate, vertical)
   1296 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1297     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:941, in DataFrame._show_string(self, n, truncate, vertical)
    923     except ValueError:
    924         raise PySparkTypeError(
    925             errorClass="NOT_EXPECTED_TYPE",
    926             messageParameters={
   (.

In [0]:
bad_path = f"{base_path}/bad_records"

df_bad = (
    spark.read
    .option("header", True)
    .option("badRecordsPath", bad_path)
    .schema(schema_clean)
    .csv(path)
)

df_bad.show()          # only good rows come back

+--------+--------+-------+-------+
|order_id|customer|product| amount|
+--------+--------+-------+-------+
|    1001|    Ravi| Laptop|55000.0|
|    1002|   Sneha| Mobile|18000.0|
|    1006|   Meera|  Watch| 9500.0|
+--------+--------+-------+-------+



In [0]:
bad_df = spark.read.json(f"{bad_path}/*/bad_records/*")
display(bad_df)

path,reason,record
dbfs:/Volumes/main/default/demo/orders.csv,"java.lang.NumberFormatException: For input string: ""twenty thousand""","1003,Arjun,Tablet,twenty thousand"
dbfs:/Volumes/main/default/demo/orders.csv,org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper,"1004,Divya,Headphones"
dbfs:/Volumes/main/default/demo/orders.csv,org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper,"1005,Karthik,Camera,32000,Express"
dbfs:/Volumes/main/default/demo/orders.csv,"java.lang.NumberFormatException: For input string: ""twenty thousand""","1003,Arjun,Tablet,twenty thousand"
dbfs:/Volumes/main/default/demo/orders.csv,org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper,"1004,Divya,Headphones"
dbfs:/Volumes/main/default/demo/orders.csv,org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper,"1005,Karthik,Camera,32000,Express"


In [0]:
#Production pattern

from pyspark.sql.functions import col

df = df_permissive   # cached, from earlier

good_df = (df.filter(col("_corrupt_record").isNull())
             .drop("_corrupt_record"))

bad_df  = df.filter(col("_corrupt_record").isNotNull())

good_df.write.mode("append").saveAsTable("main.default.orders_silver")
bad_df.write.mode("append").saveAsTable("main.default.orders_quarantine")